# 2. Train, and compare against the reference approach

The reference repository's method is DTW alignment to a reference performance. It is implemented here properly and given every advantage: three distance functions, banded and unbanded warping, a canonical or a real exemplar reference, and a per-action isotonic calibration of its distance onto the quality scale. The configuration is selected on **validation**, never on test.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt

# Notebooks run with cwd=notebooks/, so the figure and table roots have to be
# repointed at the repository's results/ tree. Without this every figure lands in
# notebooks/results/figures/ and the committed figures silently never update.
import saqa.viz as _viz, saqa.report as _report
_viz.FIGURES = ROOT / "results" / "figures"
_viz.TABLES = _report.TABLES = ROOT / "results" / "tables"
tables = _viz.TABLES
from saqa.config import load_config
from saqa.pipelines import make_splits, run_single, fit_baselines
cfg = load_config('../configs/base.yaml', ['data.num_sequences=300', 'optim.epochs=4', 'run.out_dir=../results/runs_nb'])
splits = make_splits(cfg); splits.sizes()

## Per-action calibration is worth more than the alignment

A single global isotonic map conflates 'which action' with 'how good', because a throw and a gait cycle have different spatial extents. Fixing that is the single largest change to the baseline's score, and it is the difference between a strawman and a real comparison.

In [ ]:
from saqa.baselines.fitted import DTWBaseline
from saqa.metrics import spearman
from saqa.pipelines import generator_config
g = generator_config(cfg)
rows = []
for per_action in (False, True):
    m = DTWBaseline(generator=g, per_action_calibration=per_action).fit(splits.train)
    s, _, _ = m.predict(splits.test)
    rows.append({'per_action_calibration': per_action,
                 'test_spearman': spearman(splits.test.quality, s)})
pd.DataFrame(rows).round(4)

## Train the graph model and the graph-free controls

In [ ]:
runs = {}
for arch in ('saqa_stgcn', 'tcn', 'frame_average'):
    runs[arch] = run_single(cfg, name=f'nb_{arch}', architecture=arch,
                            splits=splits, save=False, verbose=False)
    print(f"{arch:16s} rho {runs[arch].metrics['spearman']:+.4f}  "
          f"params {runs[arch].metrics['params']:.0f}")

In [ ]:
base = fit_baselines(splits, cfg, dtw_sweep=False)
rows = [{'method': k, 'spearman': spearman(splits.test.quality, v.metrics['spearman'] if False else runs[k].pred['score'])} for k in runs]
for k in ('kinematic_gbr', 'dtw_reference'):
    rows.append({'method': k, 'spearman': spearman(splits.test.quality, base[k]['score'])})
pd.DataFrame(rows).sort_values('spearman', ascending=False).round(4)

## The untrained control

A random-weights network is **not** a trivial baseline here: a random projection of movement amplitude already correlates with defect severity. Any trained model that does not clear this has learned nothing the architecture and the input statistics did not already provide.

In [ ]:
from saqa.models import build_model
from saqa.engine import predict
torch.manual_seed(999)
u = predict(build_model('saqa_stgcn'), splits.test.coords)['score']
print(f'untrained saqa_stgcn: rho {spearman(splits.test.quality, u):+.4f}')

## Committed results (from `make all`)

In [ ]:
for name in ('method_comparison', 'statistical_tests'):
    p = pathlib.Path('../results/tables') / f'{name}.csv'
    if p.exists():
        print(f'--- {name} ---'); display(pd.read_csv(p).round(4))